# Training value CNN

In [ ]:
import sys
sys.path.append("..")

import torch
from torch import nn
from torch.utils import data
from tqdm import tqdm
from matplotlib import pyplot as plt

from ai.dataset import EvalDataset
from ai.model import ValueCNN

## Hyperparameters

In [ ]:
batch_size = 256
learning_rate = 0.001
num_epochs = 10

## Data

In [ ]:
dataset = EvalDataset("../data/evals.csv", "../data/evals.pt")
train_dataset, val_dataset, test_dataset = data.random_split(dataset, [0.8, 0.1, 0.1], generator=torch.Generator().manual_seed(42))

train_loader = data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print("Train:", len(train_dataset), " val:", len(val_dataset), " test:", len(test_dataset))

## Model

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

model = ValueCNN(num_channels=7, num_filters=32, num_conv_layers=4, num_hidden=200, num_layers=4)
model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
loss_fn = nn.MSELoss()

print(model)

## Training loop

In [ ]:
def train_one_epoch(model, optimizer, train_loader, loss_fn):
    model.train()
    total_loss, num_batches = 0., 0

    for data_inputs, data_labels in tqdm(train_loader):
        data_inputs = data_inputs.to(device)
        data_labels = data_labels.to(device)

        preds = model(data_inputs)
        loss = loss_fn(preds, data_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    return total_loss / num_batches

In [ ]:
def eval_model(model, data_loader, loss_fn):
    model.eval()
    total_loss, num_batches = 0., 0
    total_abs_error, num_preds = 0., 0

    with torch.no_grad():
        for data_inputs, data_labels in tqdm(data_loader):
            data_inputs, data_labels = data_inputs.to(device), data_labels.to(device)
            preds = model(data_inputs)

            total_loss += loss_fn(preds, data_labels).item()
            num_batches += 1

            total_abs_error += (preds - data_labels).abs().sum().item()
            num_preds += data_labels.shape[0]

    return total_loss / num_batches, total_abs_error / num_preds

In [ ]:
history = {"train_loss": [], "val_loss": [], "val_mae": []}
best_val_loss = float("inf")

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, optimizer, train_loader, loss_fn)
    val_loss, val_mae = eval_model(model, val_loader, loss_fn)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_mae"].append(val_mae)
    print(f"Epoch {epoch + 1}: train loss {train_loss:.4f}, val loss {val_loss:.4f}, val MAE {val_mae:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "../models/value_cnn.tar")
        print("Checkpoint saved")

## Results

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, history["train_loss"], label="Train loss")
plt.plot(epochs, history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, history["val_mae"], label="Validation MAE")
plt.xlabel("Epoch")
plt.ylabel("MAE")
plt.legend()

plt.show()